In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

<em style="color:blue;">**Important:**</em>
For this notebook to work, you need to extract the file `mnist.zip` in the current folder and put the files mnist_train.csv and mnist_test.csv in the current folder.

# Handwritten Digit Recognition using $k$-Nearest Neighbours

This notebook uses the <em style="color:blue;">$k$-nearest neighbours algorithm</em> to recognize handwritten digits.  The digits we want to recognize
are stored as images of size $28 \times 28$ pixels.  Each pixel $p$ is stored as a number that satisfies $0 \leq p \leq 255$.  The pixel values are 
interpreted as grey values: If $p = 255$, the pixel is completely black, while $p = 0$ if the pixel is white.  The images are stored in the files `mnist_train.csv` and `mnist_test.csv`. 

The function `loadData` returns a tuple of two Datasets
$$ (\texttt{trainData}, \texttt{testData}) $$
containing the following data: 
<ul>
<li> $\texttt{X_train}$ is a matrix storing the 60,000 training images of handwritten digits.
     For each $i \in \{0,\cdots,59\,999\}$ the row $\texttt{X_train}[i]$ is an array of size $784$ storing a single image.
     </li>
<li> $\texttt{X_test}$ is a matrix containing 10,000 images of handwritten digits that can be used for testing.</li>
<li> $\texttt{Y_train}$ is an array of size 60,000. For each $i \in \{0,\cdots,59\,999\}$ the number $\texttt{Y_train}[i]$
     specifies the digit shown in the $i$th training image.
     </li>
<li> $\texttt{Y_test}$ is an array of size 10,000. For each $i \in \{0,\cdots,9\,999\}$ the number $\texttt{Y_test}[i]$
     specifies the digit shown in the $i$th test image.
     </li>
</ul>

First, we define a new type `Dataset`, which includes a 2D array `X` containing all the training images of a dataset and a 1D array `Y` holding the corresponding labels for each image.

In [ ]:
type Dataset = {
    X: number[][];
    Y: number[];
};

Then we load our test and training data from the `.csv`-files.

In [ ]:
import * as fs from 'fs';
import * as readline from 'readline';

async function loadCSV(path: string): Promise<Dataset> {
    const X: number[][] = [];
    const Y: number[] = [];

    const fileStream = fs.createReadStream(path);
    const rl = readline.createInterface({
        input: fileStream,
        crlfDelay: Infinity,
    });

    for await (const line of rl) {
        const values = line.split(',').map(Number);
        Y.push(values[0]);           
        X.push(values.slice(1));  
    }

    return { X, Y };
}

async function loadData(): Promise<[Dataset, Dataset]> {
    const trainData = await loadCSV('mnist_train.csv');
    const testData = await loadCSV('mnist_test.csv');

    return [trainData, testData];
}

In [ ]:
const [train, test] = await loadData();
const X_train = train.X;
const X_test = test.X;
const Y_train = train.Y;
const Y_test = test.Y;

Let us check what we have read:

In [ ]:
console.log('X_train array shape:', [X_train.length, X_train[0].length]); 
console.log('X_test array shape:', [X_test.length, X_test[0].length]);     
console.log('Y_train array shape:', [Y_train.length]);                     
console.log('Y_test array shape:', [Y_test.length]);  

Let us inspect the first hand written image of a digit.

In [ ]:
const firstImage = X_train[0];
console.dir(firstImage, { maxArrayLength: null });

This is an array with 784 entries.  Let us draw the corresponding picture.

The function $\texttt{showDigit}(\texttt{row}, \texttt{columns}, \texttt{offset})$ 
shows $\texttt{row} \cdot \texttt{columns}$ images of the training data.  The first image shown is the image at index $\texttt{offset}$.

In [ ]:
import { PNG } from "pngjs";
import { writeFileSync } from "fs";

function showDigit(rows: number, columns: number, offset = 0) {
  const singleWidth = 28;
  const singleHeight = 28;

  const gridWidth = columns * singleWidth;
  const gridHeight = rows * singleHeight;

  const png = new PNG({ width: gridWidth, height: gridHeight });

  for (let r = 0; r < rows; r++) {
    for (let c = 0; c < columns; c++) {
      const i = r * columns + c + offset;
      if (i >= X_train.length) continue;

      const row = X_train[i];

      for (let y = 0; y < singleHeight; y++) {
        for (let x = 0; x < singleWidth; x++) {
          const raw = row[y * singleWidth + x];
          const normalized = raw / 255; // Normalization to [0,1]
          const value = 1 - normalized;
          const v = Math.floor(value * 255);

          const gridX = c * singleWidth + x;
          const gridY = r * singleHeight + y;
          const idx = (gridY * gridWidth + gridX) << 2;

          png.data[idx + 0] = v; // R
          png.data[idx + 1] = v; // G
          png.data[idx + 2] = v; // B
          png.data[idx + 3] = 255; // Alpha
        }
      }
    }
  }

  const buffer = PNG.sync.write(png);
  writeFileSync("digits_grid.png", buffer);
}



We take a look at the first 24 images.

In [ ]:
import { readFileSync } from "fs";
import { display } from "tslab"; 

showDigit(4, 6);

const buffer = readFileSync("digits_grid.png");

const base64 = buffer.toString("base64");

const html = `<img src="data:image/png;base64,${base64}"  width="500" />`;

display.html(html);

Given two arrays $\mathbf{x}$ and $\mathbf{y}$ of the same dimension $n$, the function $\texttt{distance}(\mathbf{x}, \mathbf{y})$ computes the
<em style="color:blue;">Euclidean distance</em> between $\mathbf{x}$ and $\mathbf{y}$.  This distance is defined as follows:
$$ \sqrt{\sum\limits_{i=1}^n (x_i - y_i)^2} $$

In [ ]:
function distance(x: number[], y: number[]): number {
  if (x.length !== y.length) throw new Error("Vektoren müssen gleich lang sein");
  
  let sum = 0;
  for (let i = 0; i < x.length; i++) {
    const diff = x[i]/255 - y[i]/255; // Normalization to [0,1]
    sum += diff * diff;
  }
  return Math.sqrt(sum);
}

For example, the distance between the first two images of the training set is computed as follows:

In [ ]:
const d = distance(X_train[0], X_train[1]);
console.log(d);

The distance between the 9th and the 15th image should be smaller, because both of these images show the digit $1$ and hence these images are quite similar.
This similarity results in a smaller distance between these images.

In [ ]:
const d = distance(X_train[8], X_train[14]);
console.log(d);

Given a list $L$ of digits, the function $\texttt{maxCounts}(L)$ returns a pair $(d, p)$ where $d$ is the digit that occurs most frequently in $L$
and $p$ is the percentage of occurrences of $d$ in $L$.  For example, we have
$$ \texttt{maxCounts}([5,2,3,5,2,5,6,5,7,8]) = (5, 0.4)  $$
because the digit $5$ is the most frequent digit in the list $[5,2,3,5,2,5,6,5,7,8]$ and $40$% of the digits in this list are fives.

In [ ]:
function maxCount(L: number[]): [number, number] {
  const frequencies: Record<number, number> = {}; // number of occurrences for each digit
  let mostFrequent = L[0];                        // most frequent digit so far
  let mostFrequentCount = 1;                      // number of occurrences of most frequent digit

  for (const d of L) {
    if (frequencies[d] !== undefined) {
      frequencies[d] += 1;
    } else {
      frequencies[d] = 1;
    }

    if (frequencies[d] > mostFrequentCount) {
      mostFrequent = d;
      mostFrequentCount = frequencies[d];
    }
  }

  const percentage = mostFrequentCount / L.length;
  return [mostFrequent, percentage];
}

In [ ]:
maxCount([3, 3, 4, 2, 1, 2, 3, 2, 5, 3])

Given an image of a digit stored in the vector $\mathbf{x}$ and a number of neighbours $k$, the function $\texttt{digit}(\mathbf{x}, k)$ computes those
$k$ images in the training set `X_train` that are <em style="color:blue;">closest</em> to the image $\mathbf{x}$.  Here 
<em style="color:blue;">closeness</em> of images is defined in terms of the <em style="color:blue;">Euclidean distance</em> of the vectors that store the 
images.  From these $k$ images of the training set the function chooses the digit that occurs most frequently.  It returns a pair $(d, p)$ where $d$ is the digit that is most frequently occurring in the list of $k$ neighbours and $p$ is the percentage of images in the $k$ neighbours of $\mathbf{x}$ that show
the digit $d$.

In [ ]:
function digit(x: number[], k: number): [number, number] {
  const n = X_train.length;

  const distances: [number, number][] = [];
  for (let i = 0; i < n; i++) {
    const dist = distance(X_train[i], x);
    distances.push([dist, i]);
  }

  distances.sort((a, b) => a[0] - b[0]);

  const neighbours: number[] = distances.slice(0, k).map(([_, i]) => Y_train[i]);

  return maxCount(neighbours);
}

The function `showImage(n)` shows the $n^\mathrm{th}$ test image.

In [ ]:
import { PNG } from "pngjs";
import { display } from "tslab";

function showImage(n: number) {
  const row = X_test[n];
  const width = 28;
  const height = 28;

  const png = new PNG({ width, height });

  for (let y = 0; y < height; y++) {
    for (let x = 0; x < width; x++) {
      const raw = row[y * width + x];
      const normalized = raw / 255;  
      const value = 1 - normalized;  
      const v = Math.floor(value * 255);
      const idx = (y * width + x) << 2;
      png.data[idx + 0] = v; // R
      png.data[idx + 1] = v; // G
      png.data[idx + 2] = v; // B
      png.data[idx + 3] = 255; // Alpha
    }
  }

  const buffer = PNG.sync.write(png);
  const base64 = buffer.toString("base64");
  const html = `<img src="data:image/png;base64,${base64}" width="140" />`;

  display.html(html);
}

This function performs $k$-nearest neighbour classification for the $n$-th image of the test set.  It also prints the image.

In [ ]:
async function test(n: number, k: number) {
  console.log(`Testing image ${n}:`);
    
  showImage(n);

  const [d, p] = digit(X_test[n], k);

  console.log(`I believe with a certainty of ${(p * 100).toFixed(2)}% that the image shows the digit ${d}.`);
}


In [ ]:
test(2, 13)

Let us classify the first 20 images from the test set.

In [ ]:
console.time("test20");

for (let n = 0; n < 20; n++) {
  await test(n, 13); 
}

console.timeEnd("test20");

In [ ]:
async function check(n: number, k: number): Promise<number> {
  const [d, p] = digit(X_test[n], k);

  if (d === Y_test[n]) {
    return 0;
  } else {
    console.log(`\nImage number ${n} wrongly identified: I guessed a ${d}, but it's a ${Y_test[n]}.`);
    await showImage(n);
    return 1;
  }
}


In [ ]:
console.time("test10000");

let errors = 0;

for (let n = 0; n < 10000; n++) {
  errors += await check(n, 7);
}

console.timeEnd("test10000");
console.log(`There were ${errors} errors out of 10000 images.`);